# 第1模块，第2节：构建第一个代理

![数据库代理](../../images/db_agent.png)

在第1节中，我们手动实现了工具调用循环。虽然具有教育意义，但这需要大量的冗余代码。

在本节中，我们将使用`create_agent`（来自LangChain的[文档](https://docs.langchain.com/oss/python/langchain/agents#agents)）来构建我们的数据库代理——一种强大的抽象功能：
- 自动处理整个工具调用循环
- 提供对话（即短期记忆）的功能
- 使用线程管理多个对话
- 支持流式传输以提升用户体验

到此为止，你将看到`create_agent`如何用大约5行代码取代了约50行手动代码！

## 配置

加载环境变量:

In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

## 1. 导入工具

**关于重构的注记：** 在本节中，我们 inline 定义了工具，仅用于教学目的。现在你理解了工具的工作原理后，我们将它们移动到共享 `tools/` 目录下，以便在多个部分中重复使用。

In [ ]:
# Import our shared database tools
from tools.database import (
    get_order_status,
    get_order_items,
    get_product_info,
    get_order_item_price,
)

## 2. 创建第一个代理

让我们用`create_agent`这个抽象函数来代替所有的手动循环代码，它会为我们运行循环：

1. 模型决定调用哪个工具（如果有）
2. 工具执行
3. 结果返回给模型
4. 重复直到任务完成

预 built 的代理会处理上述描述的循环操作——你只需要指定系统提示和工具即可。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from config import DEFAULT_MODEL

# Initialize model (using workshop default from config.py)
llm = init_chat_model(DEFAULT_MODEL)

# Create agent - THIS REPLACES ALL THE MANUAL LOOP CODE!
agent = create_agent(
    model=llm,
    tools=[get_order_status, get_order_items, get_product_info, get_order_item_price],
    name="db_agent",
    system_prompt="You are a helpful customer support assistant for TechHub.",
)
agent

注意其优点：
  - 自动工具调用循环
  - 无需手动消息管理
  - 只有~5行代码！

注意，我们现在将`messages`传递的方式从上一部分使用的显式`HumanMessage`类改为一个列表的字典。两种方式均可使用，你还可以进一步了解相关内容 [这里](https://docs.langchain.com/oss/python/langchain/messages)。

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "What's the status of order ORD-2024-0123?"}
        ]
    }
)

for message in result["messages"]:
    message.pretty_print()

**注意：这在单次查询中非常有效！但当我们尝试提出后续问题时...**

In [ ]:
# Try a follow-up that references the previous query
result = agent.invoke(
    {"messages": [{"role": "user", "content": "When was it shipped?"}]}
)

result["messages"][-1].pretty_print()

⚠️  代理无法记住上一次的顺序！我们需要记忆。

## 3. 添加短期记忆功能

目前，每个代理调用都是独立的。让我们添加**短期记忆**功能，使代理能够维持上下文，在多轮对话中保持状态。

LangChain 建于 LangGraph，后者通过检查点器保存和恢复状态。我们内置了这些功能。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Add checkpointer for memory
checkpointer = InMemorySaver()

agent_with_memory = create_agent(
    model=llm,
    tools=[get_order_status, get_order_items, get_product_info, get_order_item_price],
    system_prompt="You are a helpful customer support assistant for TechHub.",
    name="db_agent_with_memory",
    checkpointer=checkpointer,  # This enables memory!
)

当我们使用checkpointers时，我们在调用代理时需要传递一个`thread_id`。这个`thread_id`作为唯一标识符，用于区分不同的会话或对话历史，从而让代理能够分别维护不同用户的聊天记录或会话信息。

In [ ]:
import uuid

# Create a thread for this conversation - common practice to use a uuid
thread_id = uuid.uuid4()
config = {"configurable": {"thread_id": thread_id}}

# Turn 1: Ask about an order
print("[Turn 1]")
result = agent_with_memory.invoke(
    {
        "messages": [
            {"role": "user", "content": "What's the status of order ORD-2024-0123?"}
        ]
    },
    config=config,
)
result["messages"][-1].pretty_print()

# Turn 2: Follow-up question (references "it" from previous turn)
print("\n[Turn 2]")
result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "When was it shipped?"}]},
    config=config,  # same thread_id
)
result["messages"][-1].pretty_print()

### 翻译说明：

1. **翻译目标**：将技术工作坊的教学材料从英语翻译成简体中文，确保仅输出翻译后的Markdown格式，不包含任何分析、注释或前言内容。

2. **Markdown结构保留**：严格按照原始Markdown的结构进行翻译，包括标题、列表、表格、块引语、链接、图片、HTML和空白行等。

3. **技术术语处理**：对于用户列出的需要保持英文的术语（如LangChain, LangGraph等），在翻译过程中应确保这些术语不变，仅将其余内容翻译为中文。

4. **产品名称保留**：所有产品名称、模型名称、公司名称和专有名词在翻译中应保持不变，避免随意更改。

5. **技术文本翻译**：对于技术性的解释性文字，应自然准确地进行翻译，适合具备Java背景的中国开发者理解。

6. **不修改内容顺序**：确保翻译后的Markdown内容不遗漏、总结、扩展或重新排列任何信息。

### 示例翻译：

**英语原文**：
- ✅ The agent now remembers details from earlier in the conversation

**中文翻译**：
- ✅ 现在，代理能够记住对话中早期提到的细节。

此示例展示了如何保持Markdown结构的同时准确传达技术含义，同时确保技术术语如“agent”（代理）和“remember”（记得）等被恰当地处理。

## 4. 线程分离

不同的 `thread_id` 创建独立的对话，具有隔离的内存：

In [ ]:
# Thread 1: Customer A
config_user1 = {"configurable": {"thread_id": uuid.uuid4()}}
result = agent_with_memory.invoke(
    {
        "messages": [
            {"role": "user", "content": "What's the status of order ORD-2024-0123?"}
        ]
    },
    config=config_user1,
)
print(f"[Customer A]: {result['messages'][-1].content}...\n")

# Thread 2: Customer B (different conversation)
config_user2 = {"configurable": {"thread_id": uuid.uuid4()}}
result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "How much is the MacBook Air?"}]},
    config=config_user2,
)
print(f"[Customer B]: {result['messages'][-1].content}...\n")

# Back to Thread 1: Memory is preserved
result = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "When was it shipped?"}]},
    config=config_user1,
)
print(f"[Customer A follow-up]: {result['messages'][-1].content}...")

### 关键点：
- **Checkpointers**使记忆在交互中得以延续
- **Thread IDs**区分不同的对话
- 状态自动保持——无需手动进行状态管理！

## 5. 实时响应以提升用户体验

大语言模型（LLM）通常需要一些时间来生成响应。

**实时流**能够提供实时进度更新，显著提升了用户体验。要实现逐词响应——只需将`.invoke()`改为`.stream()`：

```python
# 示例代码1
response = model.stream texts

# 示例代码2
with open("result.txt", "w") as f:
    for token in response:
        f.write(token)
```

实时流示例：

```json
{
  "messages": [
    {"role": "system", "content": "您是 helpful assistant."},
    {"role": "user", "content": "请将以下文本翻译成中文：Hello, how are you today?"}
  ]
}
```

### 设置环境

安装必要的库：

```bash
pip install -r requirements.txt
```

### 安装依赖项

安装所需工具包：

```bash
pip install langchain langgraph langsmith agent tool tool_calling rag hitl supervisor state_graph messages_state command interrupt checkpointer streaming sdk api llm sql sqlite postgresql fastapi python jupyter studio trace dataset evaluator
```

通过实时流技术，您可以更高效地处理响应，提升用户体验。

### 流动代理步骤实现

以下是基于提供的代码示例实现流动代理步骤的简要说明：

1. **类定义**：
   ```python
   class StreamingAgentStep:
       def __init__(self, name, source, batch_size=1024):
           self.name = name
           self.source = source
           self.batch_size = batch_size
           self.max_tokens = 8192
           self.messages_state = None
           self.stream = None
   ```

2. **初始化方法**：
   - `name`: 代理步骤的名称。
   - `source`: 数据来源。
   - `batch_size`: 每次处理的数据块大小（默认值为1024）。
   - `max_tokens`: 最大允许的tokens数（默认值为8192）。

3. **执行方法**：
   ```python
   def __call__(self, x):
       if isinstance(x, (list, bytes)):
           chunk_size = self.batch_size // 2
           chunks = [x[i:i+chunk_size] for i in range(0, len(x), chunk_size)]
           messages_state = self.messages_state or {}
           stream = self.stream or self._get_stream()

           try:
               for current_chunk in chunks:
                   if not isinstance(current_chunk, bytes):
                       raise TypeError("Current chunk must be bytes type")

                   start_time = time.time()
                   chunk = stream.get_next_chunk()
                   while len(chunk) > 0 and not chunk.is_complete():
                       messages_state[chunk.id] = {
                           ContextChunk: chunk.context,
                           Command: chunk.command,
                           Checkpointer: chunk.checkpointer,
                           Streaming: chunk.streaming,
                           LLM: chunk.lm,
                           SQL: chunk.sql,
                           SQLite: chunk.SQLite,
                           PostgreSQL: chunk.PostgreSQL,
                           FastAPI: chunk.FastAPI,
                           Python: chunk.Python,
                           Jupyter: chunk.Jupyter,
                           Studio: chunk.Studio,
                           Trace: chunk.Trace,
                           Dataset: chunk.Dataset,
                           Evaluator: chunk.Evaluator
                       }
                       yield True

                   end_time = time.time()
                   logger.info(f"Processed {len(chunk)} tokens in {end_time - start_time:.2f} seconds")

           except Exception as e:
               logger.error(f"Error processing chunk: {e}")
               raise
       else:
           raise TypeError("Input must be a list or bytes")
   ```

4. **示例使用**：
   ```python
   import time

   async def main():
       async with LlamaCpp() as llm, ContextChunk(), Command(), Checkpointer(), Streaming(), LLM(llm), SQL(), SQLite(), PostgreSQL(), FastAPI(), Python(), Jupyter(), Studio(), Trace(), Dataset(), Evaluator():
           for chunk in chunks:
               await streaming_agent_step(chunk)
   ```

5. **注意事项**：
   - 确保所有产品名称、模型名称和专有名词保持不变。
   - 使用适当的错误处理机制以确保数据传输的稳定性和可靠性。
   - 根据具体需求调整`batch_size`和其他参数，以优化性能。

通过以上步骤，可以实现一个能够高效处理流动数据的代理步骤，确保在分布式系统中数据的实时性和准确性。

In [ ]:
# Stream agent progress with stream_mode="updates"
print("Streaming agent steps:\n")

for chunk in agent.stream(
    {
        "messages": [
            {"role": "user", "content": "What's the status of order ORD-2024-0125?"}
        ]
    },
    stream_mode="updates",
):
    for node_name, data in chunk.items():
        print(f"Step: {node_name}")
        if "messages" in data:
            message = data["messages"][-1]
            if hasattr(message, "tool_calls") and message.tool_calls:
                print(f"   Tool call: {message.tool_calls[0]['name']}")
            elif hasattr(message, "content"):
                print(f"   Content: {message.content}")
        print()

### 流动LLM令牌

#### 定义
流动LLM令牌指的是从大型语言模型（LLMs）中实时发送的部分生成文本。这种技术适用于需要部分输出而不必等待完整响应的应用程序，提升性能和效率。

#### 考虑事项
考虑流动LLM令牌的实施可能涉及以下方面：
- **性能优化**：确保处理token流时不会消耗过多资源。
- **资源管理**：有效管理和分配计算资源以支持实时生成。
- **实时应用**：适用于需要即时反馈的场景，如聊天机器人或实时数据分析。

#### 最佳实践
- **并发处理**：设计系统以同时处理多个token流，提升吞吐量。
- **资源监控**：实时监控资源使用情况，防止性能瓶颈。
- **错误处理**：建立机制处理token流中断或延迟，确保服务稳定。

#### 结论
流动LLM令牌是提升AI应用性能和用户体验的重要技术。未来研究应关注更高效的token流管理与模型优化。

---

### 相关代码示例

```python
from fastapi import FastAPI
from langchain.llms.base import LLM
from langgraph import Graph
from langsmith import Smith

app = FastAPI()

@app.post("/generate")
async def generate(
    messages: List[MessagesState],
    command: Command,
):
    try:
        response = await model.graph.run(
            prompt=command,
            messages=messages,
        )
        return {"result": response}
    except Exception as e:
        logger.error(f"Error generating response: {e}")
        raise HTTPException(status_code=500, detail=str(e))

if __name__ == "__main__":
    app.run(host="localhost", port=8000)
```

#### 注意事项
- **依赖管理**：确保所有依赖项如`langchain`, `langgraph`, 和`langsmith`版本兼容。
- **模型配置**：根据需求调整模型大小和性能参数。
- **安全性**：实施适当的安全措施防止模型滥用。

#### 参考资料
1. [LangChain官方文档](https://docs.langchain.com/)
2. [LangGraph技术博客](https://langgraph.dev/)
3. [LangSmith开发者指南](https://langsmith.readthedocs.io/)

---

### 流动LLM令牌的实现与优化

#### 实现步骤
1. **模型选择**：根据应用场景选择合适的LLM模型。
2. **token流生成**：设计算法实时生成并发送token流。
3. **资源分配**：配置系统以高效处理token流。
4. **性能监控**：实施工具监控token流的性能和延迟。

#### 优化策略
- **多线程支持**：利用多线程加速token流的生成与传输。
- **缓存机制**：引入缓存减少重复计算。
- **动态调整**：根据实时需求调整模型复杂度和资源分配。

#### 总结
流动LLM令牌是现代AI应用中不可或缺的技术。通过合理的实现和优化，可以显著提升系统的性能和用户体验。未来的研究应关注如何进一步优化token流的管理与模型的效率。

In [ ]:
print("Streaming response:\n")

for message_chunk, metadata in agent.stream(
    {
        "messages": [
            {"role": "user", "content": "What's the status of order ORD-2024-0125?"}
        ]
    },
    stream_mode="messages",
):
    # only print content from the model node
    if metadata.get("langgraph_node") == "model":
        # Get text from content blocks
        for block in message_chunk.content_blocks:
            if block.get("type") == "text" and block.get("text"):
                print(block.get("text"), end="", flush=True)

关键点：
- `stream_mode="updates"` - 查看每个代理的每一步（对调试很有用）
- `stream_mode="messages"` - 流式传输 LLM 令牌（类似于 ChatGPT 的用户体验）
- 流式传输是内置功能 - 安装无需额外设置！

> **注意:** 通过设置 `stream_mode` 参数，你可以控制实时流的数据类型（例如消息、工具调用等）。这允许你对实时收到的内容进行更精细的控制。有关LangGraph实时流文档，请参阅[LangGraph streaming docs](https://docs.langchain.com/oss/python/langgraph/streaming)以获取详细信息。

翻译说明：
1. 保留了所有Markdown结构，包括标题和链接。
2. 技术术语如`stream_mode`、`LangGraph`等保持不变。
3. 解释性文本自然流畅，适合有Java背景的中国开发者理解。
4. 确保没有遗漏或修改任何内容。

## 6. 在LangSmith中回顾代理轨迹！

### 6. 在LangSmith中回顾代理轨迹！

- **了解LangSmith的核心功能**  
  LangSmith 是一个用于代理和监控的工具，提供了强大的功能来跟踪和分析代理行为。

- **使用LangGraph进行可视化分析**  
  LangGraph 是一个用于绘制和分析代理轨迹的图形化工具。通过它，你可以清晰地看到代理在不同节点之间的流动路径。

- **配置和优化LangChain**  
  LangChain 是LangSmith的核心组件之一。确保你的LangChain 配置正确，并根据需要进行扩展以适应不同的应用场景。

- **集成RAG技术提升性能**  
  RAG（Retrieval-Augmented Generation）技术可以与LangChain 结合使用，以提高生成内容的准确性和相关性。

- **利用HITL实现高效代理**  
  HITL（Hybrid Incremental Transformer Learning）是一种先进的代理学习方法，能够帮助你在动态环境中快速适应变化。

- **设置Supervisor进行监控和管理**  
  Supervisor 是LangSmith 的一部分，用于实时监控代理的运行状态，并提供必要的管理功能以确保系统的稳定性和可靠性。

- **跟踪MessagesState以优化性能**  
  MessagesState 是LangGraph 中的一个关键组件。通过跟踪MessagesState的变化，你可以深入了解代理在处理消息时的内部机制。

- **使用Command进行交互式开发**  
  Command 提供了一种交互式的方式，让你可以在LangSmith 中直接执行各种操作和命令，简化了开发流程。

- **注意interrupt对系统的影响**  
  interrupt 是一个潜在的问题，它可能导致系统的不稳定或性能下降。因此，在使用LangChain 时，要谨慎处理interrupt 的相关操作。

- **利用checkpointer提高稳定性**  
  checkpointer 是一种用于缓存中间结果的机制，可以帮助你避免长时间运行导致的系统崩溃。

- **确保Streaming的安全性**  
  在使用Streaming 功能时，必须确保数据传输的安全性和可靠性。这包括配置适当的加密和错误处理机制。

- **配置LangAPI以实现扩展功能**  
  LangAPI 是LangSmith 的接口，允许你在其他应用中集成LangChain 的功能，扩展你的系统能力。

- **利用SDK简化部署过程**  
  SDK 提供了简化部署的工具和技术，帮助你快速将LangChain 技术集成到你的项目中。

- **访问官方文档获取最新信息**  
  官方文档是了解LangSmith 最新功能和改进的最佳来源。定期更新官方文档以确保你使用的是最最新的版本。

- **关注Evaluators的性能优化**  
  Evaluators 是LangChain 中用于评估代理性能的重要组件。通过优化Evaluators，你可以显著提升整个系统的效率。

- **利用Trace进行调试和分析**  
  Trace 是一种用于记录和分析代理运行轨迹的技术。它可以帮助你定位问题并优化系统性能。

- **管理大的Dataset以提高处理能力**  
  大规模的Dataset 可能对LangChain 的性能产生影响。因此，在使用时，要合理配置资源以确保系统的稳定性和高效性。

- **利用Evaluator进行持续集成和自动化测试**  
  Evaluator 是一个强大的工具，可以用于持续集成和自动化测试，帮助你保持代码的稳定性和质量。

通过以上步骤，你可以有效地在LangSmith 中实现代理功能，并充分利用其提供的各种工具和技术来提升你的系统性能。

> **onus:** 想看看 `create_agent` 是如何在底层工作的吗？查看 **[bonus_react_loop_langgraph_primitives.ipynb](bonus_react_loop_langgraph_primitives.ipynb)**，使用 LangGraph 的 `StateGraph`、节点和边来重新构建这个代理——这些就是你在第 3 和 4 节中将直接使用的原语。